Data processing

In [ ]:
import os
import pandas as pd
import shutil
from io import StringIO


def process_pos_files(input_folder, output_folder):
    """
    Batch process .pos files: delete the first 36 lines -> format -> convert to CSV

    :param input_folder: Path to the input folder containing the .pos files to be processed
    :param output_folder: Path to the output folder used to save the processed files
    :return: Number of successfully processed files
    """
    # Ensure the output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    success_count = 0

    # Iterate through all .pos files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith('.pos'):
            input_file = os.path.join(input_folder, filename)
            output_filename = os.path.splitext(filename)[0] + '.csv'
            output_file = os.path.join(output_folder, output_filename)

            try:
                # Step 1: Delete the first 36 lines and format the data
                temp_lines = []
                with open(input_file, 'r') as infile:
                    # Skip the first 36 lines
                    for _ in range(36):
                        next(infile, None)

                    # Read and format the remaining lines
                    for line in infile:
                        stripped_line = line.strip()
                        if stripped_line:
                            # Replace multiple spaces or tabs with a single space
                            formatted_line = ' '.join(stripped_line.split())
                            temp_lines.append(formatted_line)

                # Step 2: Convert to CSV
                if temp_lines:
                    # Write the formatted data to a temporary string
                    temp_content = '\n'.join(temp_lines)

                    # Read with pandas and save as CSV
                    df = pd.read_csv(StringIO(temp_content), delimiter=' ', header=None)
                    df.to_csv(output_file, index=False, header=False)

                    success_count += 1
                    print(f"POS file processed successfully: {filename} -> {output_filename}")
                else:
                    print(f"Warning: File {filename} is empty after processing")

            except Exception as e:
                print(f"Error processing POS file {filename}: {e}")

    print(f"Successfully processed {success_count} POS files")
    return success_count


def extract_date_range(input_file, output_file, start_date, end_date):
    """
    Extract rows within the specified date range from a CSV file and retain only the date and coordinate columns.

    :param input_file: Path to the input CSV file
    :param output_file: Path to the output CSV file
    :param start_date: Start date (format: YYYYMMDD)
    :param end_date: End date (format: YYYYMMDD)
    :return: Return True if processing succeeds, otherwise return False
    """
    try:
        # Read the CSV file
        df = pd.read_csv(input_file)

        # Ensure the date column exists and convert it to a string
        if '*YYYYMMDD' not in df.columns:
            raise ValueError("The CSV file is missing the '*YYYYMMDD' column")

        # Convert the date column to datetime format for comparison
        df['*YYYYMMDD'] = pd.to_datetime(df['*YYYYMMDD'], format='%Y%m%d')

        # Convert the input dates to datetime format
        start_date = pd.to_datetime(start_date, format='%Y%m%d')
        end_date = pd.to_datetime(end_date, format='%Y%m%d')

        # Extract rows within the specified date range
        mask = (df['*YYYYMMDD'] >= start_date) & (df['*YYYYMMDD'] <= end_date)
        filtered_df = df[mask]

        # Retain only the required columns (date and coordinates)
        required_columns = ['*YYYYMMDD', 'NLat', 'Elong', 'dN', 'dE']
        # Check whether all required columns exist
        missing_cols = [col for col in required_columns if col not in filtered_df.columns]
        if missing_cols:
            raise ValueError(f"The CSV file is missing the following columns: {missing_cols}")

        filtered_df = filtered_df[required_columns]

        # Convert the date back to YYYYMMDD format
        filtered_df['*YYYYMMDD'] = filtered_df['*YYYYMMDD'].dt.strftime('%Y%m%d')

        # Save the extracted data to a new CSV file
        filtered_df.to_csv(output_file, index=False)
        print(f"Date range extracted successfully: {os.path.basename(input_file)} -> {os.path.basename(output_file)}")

        return True
    except Exception as e:
        print(f"Error extracting date range from {os.path.basename(input_file)}: {e}")
        return False


def rename_first_column_and_save(input_folder, output_folder):
    """
    Rename the first column of the CSV file to YYYYMMDD

    :param input_folder: Path to the input folder
    :param output_folder: Path to the output folder
    :return: Number of successfully processed files
    """
    # Ensure the output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    success_count = 0

    # Iterate through all CSV files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith('.csv'):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)

            try:
                # Read the CSV file
                df = pd.read_csv(input_path)

                # Check whether there are columns
                if not df.empty:
                    # Get the name of the first column
                    first_col = df.columns[0]

                    # Rename the first column to YYYYMMDD
                    df.rename(columns={first_col: 'YYYYMMDD'}, inplace=True)

                    # Save to the new folder
                    df.to_csv(output_path, index=False)
                    success_count += 1
                    print(f"Column renamed successfully: {filename}")
            except Exception as e:
                print(f"Error renaming column in {filename}: {e}")

    print(f"Successfully renamed the columns of {success_count} files")
    return success_count


def rename_and_save_csv(source_folder, target_folder):
    """
    Rename CSV files, retaining only the first 4 characters of each filename

    :param source_folder: Path to the source folder
    :param target_folder: Path to the target folder
    :return: Number of successfully processed files
    """
    # Create the target folder if it does not exist
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)

    files = os.listdir(source_folder)

    # Filter CSV files
    csv_files = [f for f in files if f.lower().endswith('.csv')]

    success_count = 0

    # Iterate through and process each CSV file
    for filename in csv_files:
        try:
            # Get the filename without the extension
            name_without_ext = os.path.splitext(filename)[0]

            # Take the first 4 characters (retain all if fewer than 4)
            new_name = name_without_ext[:4] if len(name_without_ext) >= 4 else name_without_ext

            # Construct the new filename (retain the .csv extension)
            new_filename = f"{new_name}.csv"

            # Construct the full paths
            old_path = os.path.join(source_folder, filename)
            new_path = os.path.join(target_folder, new_filename)

            # Avoid filename conflicts (add a sequence number if a file with the same name already exists)
            counter = 1
            while os.path.exists(new_path):
                new_filename = f"{new_name}_{counter}.csv"
                new_path = os.path.join(target_folder, new_filename)
                counter += 1

            # Copy the file to the new folder (retain the original file)
            shutil.copy2(old_path, new_path)
            success_count += 1
            print(f"File renamed successfully: {filename} -> {new_filename}")
        except Exception as e:
            print(f"Error renaming file {filename}: {e}")

    print(f"Successfully renamed {success_count} files")
    return success_count


def batch_extract_date_range(input_folder, output_folder, start_date, end_date):
    """
    Batch process all CSV files in the folder and extract rows within the specified date range.

    :param input_folder: Path to the input folder containing the CSV files to be processed
    :param output_folder: Path to the output folder used to save the extracted CSV files
    :param start_date: Start date (format: YYYYMMDD)
    :param end_date: End date (format: YYYYMMDD)
    :return: Number of successfully processed files
    """
    # Ensure the output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    success_count = 0

    # Iterate through all files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith('.csv'):  # Process only CSV files
            input_file = os.path.join(input_folder, filename)
            # Construct the output file path
            output_filename = f"{os.path.splitext(filename)[0]}_filtered.csv"
            output_file = os.path.join(output_folder, output_filename)
            # Extract data
            if extract_date_range(input_file, output_file, start_date, end_date):
                success_count += 1

    print(f"Successfully extracted date-range data from {success_count} files")
    return success_count


def process_all_steps(input_folder, output_base_folder, start_date, end_date, process_pos=False):
    """
    Execute the complete data processing workflow

    :param input_folder: Original input folder
    :param output_base_folder: Base output folder
    :param start_date: Start date
    :param end_date: End date
    :param process_pos: Whether to process POS files first
    :return: Processing result dictionary
    """
    # Define the output folders for each step
    if process_pos:
        pos_output_folder = os.path.join(output_base_folder, "pos_processed")
        step1_folder = os.path.join(output_base_folder, "step1_date_extracted")
        step2_folder = os.path.join(output_base_folder, "step2_renamed_columns")
        step3_folder = os.path.join(output_base_folder, "step3_renamed_files")
    else:
        pos_output_folder = None
        step1_folder = os.path.join(output_base_folder, "step1_date_extracted")
        step2_folder = os.path.join(output_base_folder, "step2_renamed_columns")
        step3_folder = os.path.join(output_base_folder, "step3_renamed_files")

    print("Starting the data processing workflow...")
    print("=" * 50)

    counts = {}

    # Step 0: Process POS files (if needed)
    if process_pos:
        print("Step 0: Process POS files")
        counts['pos'] = process_pos_files(input_folder, pos_output_folder)
        # Use the folder containing the processed POS files as input for subsequent steps
        input_folder = pos_output_folder

    # Step 1: Extract date-range data
    print("\nStep 1: Extract data within the specified date range")
    counts['step1'] = batch_extract_date_range(input_folder, step1_folder, start_date, end_date)

    # Step 2: Rename the first column
    print("\nStep 2: Rename the first column to YYYYMMDD")
    counts['step2'] = rename_first_column_and_save(step1_folder, step2_folder)

    # Step 3: Rename files
    print("\nStep 3: Rename files (retain the first 4 characters)")
    counts['step3'] = rename_and_save_csv(step2_folder, step3_folder)

    print("=" * 50)
    print("Data processing workflow completed!")
    
    if process_pos:
        print(f"POS files processed: {counts['pos']} files")
    print(f"Step 1 successfully processed: {counts['step1']} files")
    print(f"Step 2 successfully processed: {counts['step2']} files")
    print(f"Step 3 successfully processed: {counts['step3']} files")

    result = {
        'step1_folder': step1_folder,
        'step2_folder': step2_folder,
        'step3_folder': step3_folder,
        'counts': counts
    }
    
    if process_pos:
        result['pos_folder'] = pos_output_folder

    return result


# Main program entry point
if __name__ == "__main__":
    # Configuration parameters
    input_folder = 'D://a//master//Earthquake-US//Data//20190706-500//PosData'
    output_base_folder = 'D://a//master//Earthquake-US//Data//20190706-500//ProcessedData'
    start_date = '20180706'
    end_date = '20200706'
    
    # Whether to process POS files (set to True if the input files are .pos files; set to False if they are CSV files)
    process_pos_files_flag = True 

    # Execute the complete processing workflow
    result = process_all_steps(input_folder, output_base_folder, start_date, end_date, process_pos_files_flag)

    print(f"\nFinal processing results are saved in: {result['step3_folder']}")

Data check

In [ ]:
import os
import pandas as pd
from datetime import timedelta

def check_date_continuity(csv_file):
    """
    Check whether the dates in the YYYYMMDD column of a single CSV file are continuous

    :param csv_file: CSV file path
    :return:
        is_continuous: Whether the dates are continuous
        missing_dates: List of missing dates
        total_days: Actual number of existing days
        expected_days: Theoretical total number of days
        missing_ratio: Proportion of missing dates
        date_range: Date range
        error: Error message
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_file)

        # Check whether the YYYYMMDD column exists
        if 'YYYYMMDD' not in df.columns:
            return False, [], 0, 0, 0, "", f"Error: 'YYYYMMDD' column not found in {os.path.basename(csv_file)}"

        # Convert to datetime format and sort
        dates = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d').sort_values().reset_index(drop=True)

        # If the file is empty
        if len(dates) == 0:
            return False, [], 0, 0, 0, "", f"Error: no valid dates in {os.path.basename(csv_file)}"

        # Remove duplicate dates to prevent them from affecting the continuity check
        dates = dates.drop_duplicates().reset_index(drop=True)

        # Record missing dates
        missing_dates = []

        for i in range(1, len(dates)):
            delta = (dates.iloc[i] - dates.iloc[i - 1]).days

            if delta > 1:
                missing_start = dates.iloc[i - 1] + timedelta(days=1)
                missing_end = dates.iloc[i] - timedelta(days=1)

                current = missing_start
                while current <= missing_end:
                    missing_dates.append(current.strftime('%Y%m%d'))
                    current += timedelta(days=1)

        # Actual number of existing days
        total_days = len(dates)

        # Theoretical total number of days: complete continuous days from the earliest date to the latest date
        expected_days = (dates.max() - dates.min()).days + 1

        # Number of missing days
        missing_count = len(missing_dates)

        # Missing ratio
        missing_ratio = missing_count / expected_days if expected_days > 0 else 0

        # Whether the dates are continuous
        is_continuous = missing_count == 0

        # Date range
        date_range = f"{dates.min().strftime('%Y%m%d')} to {dates.max().strftime('%Y%m%d')}"

        return is_continuous, missing_dates, total_days, expected_days, missing_ratio, date_range, None

    except Exception as e:
        return False, [], 0, 0, 0, "", f"Error processing {os.path.basename(csv_file)}: {str(e)}"


def batch_check_date_continuity(folder_path, output_file=None):
    """
    Batch-check the date continuity of all CSV files in a folder

    :param folder_path: Folder path containing CSV files
    :param output_file: Result output file path, optional
    """
    results = []

    # Used to calculate the overall missing ratio for all files
    total_missing_count_all = 0
    total_expected_days_all = 0
    total_actual_days_all = 0

    # Iterate through all CSV files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)

            is_continuous, missing_dates, total_days, expected_days, missing_ratio, date_range, error = check_date_continuity(file_path)

            if error:
                results.append({
                    'Filename': filename,
                    'Status': 'Error',
                    'Message': error
                })
            else:
                missing_count = len(missing_dates)

                status = 'Continuous' if is_continuous else f'Discontinuous ({missing_count} missing)'

                results.append({
                    'Filename': filename,
                    'Status': status,
                    'Actual Days': total_days,
                    'Expected Days': expected_days,
                    'Missing Count': missing_count,
                    'Missing Ratio': missing_ratio,
                    'Missing Ratio (%)': missing_ratio * 100,
                    'Date Range': date_range,
                    'Missing Dates': ', '.join(missing_dates) if missing_dates else 'None'
                })

                # Accumulate overall statistics
                total_missing_count_all += missing_count
                total_expected_days_all += expected_days
                total_actual_days_all += total_days

    # Calculate the overall missing ratio for all files
    total_missing_ratio_all = (
        total_missing_count_all / total_expected_days_all
        if total_expected_days_all > 0 else 0
    )

    # Print results
    print("\nCheck Results:")
    print("-" * 100)

    for result in results:
        if 'Message' in result:
            print(f"File: {result['Filename']} - {result['Message']}")
        else:
            print(f"File: {result['Filename']}")
            print(f"Status: {result['Status']}")
            print(f"Actual Days: {result['Actual Days']}")
            print(f"Expected Days: {result['Expected Days']}")
            print(f"Missing Days: {result['Missing Count']}")
            print(f"Missing Ratio: {result['Missing Ratio (%)']:.4f}%")
            print(f"Date Range: {result['Date Range']}")

            if 'Discontinuous' in result['Status']:
                print(f"Missing Dates: {result['Missing Dates']}")

            print("-" * 100)

    # Print overall statistics
    print("\nOverall Statistics:")
    print("=" * 100)
    print(f"Total Actual Days for All Files: {total_actual_days_all}")
    print(f"Total Expected Days for All Files: {total_expected_days_all}")
    print(f"Total Missing Days for All Files: {total_missing_count_all}")
    print(f"Overall Missing Ratio for All Files: {total_missing_ratio_all * 100:.4f}%")
    print("=" * 100)

    # Save to file
    if output_file:
        result_df = pd.DataFrame(results)

        # Add an overall summary row at the end
        summary_row = {
            'Filename': 'ALL_FILES_SUMMARY',
            'Status': 'Summary',
            'Actual Days': total_actual_days_all,
            'Expected Days': total_expected_days_all,
            'Missing Count': total_missing_count_all,
            'Missing Ratio': total_missing_ratio_all,
            'Missing Ratio (%)': total_missing_ratio_all * 100,
            'Date Range': 'All files',
            'Missing Dates': 'See individual files'
        }

        result_df = pd.concat(
            [result_df, pd.DataFrame([summary_row])],
            ignore_index=True
        )

        result_df.to_csv(output_file, index=False, encoding='utf-8-sig')

        print(f"\nResults saved to: {output_file}")


folder_path = r'D://a//master//Earthquake-US//Data//20190706-102//PosData-6'

output_file = r'D://a//master//Earthquake-US//Fig-Over-Output//date_continuity_check_results.csv'

batch_check_date_continuity(folder_path, output_file)

Fill in the missing values

In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta

def process_csv_files(input_folder, output_folder):
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Define the date range 
    start_date = datetime(2018, 7, 6)
    end_date = datetime(2020, 7, 6)
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    expected_dates = [int(date.strftime('%Y%m%d')) for date in date_range]
    
    # Process each CSV file in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith('.csv'):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            
            # Read the CSV file
            df = pd.read_csv(input_path)
            
            # Convert YYYYMMDD to datetime for sorting
            df['Date'] = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d')
            df = df.sort_values('Date')
            
            # Create a complete date range DataFrame
            complete_df = pd.DataFrame({'Date': date_range})
            complete_df['YYYYMMDD'] = complete_df['Date'].dt.strftime('%Y%m%d').astype(int)
            
            # Merge with original data
            merged_df = pd.merge(complete_df, df, on='YYYYMMDD', how='left')
            
            # Drop the extra Date columns and keep only one
            merged_df = merged_df.drop(columns=['Date_y'])
            merged_df = merged_df.rename(columns={'Date_x': 'Date'})
            
            # Get numeric columns for interpolation (excluding YYYYMMDD, Date, and any other non-numeric columns)
            numeric_cols = merged_df.select_dtypes(include=['float64', 'int64']).columns
            numeric_cols = [col for col in numeric_cols if col not in ['YYYYMMDD']]
            
            # Apply linear interpolation to numeric columns
            merged_df[numeric_cols] = merged_df[numeric_cols].interpolate(method='linear')
            
            # Reorder columns to match original (YYYYMMDD first)
            original_columns = df.columns.tolist()
            merged_df = merged_df[original_columns]
            
            # Ensure YYYYMMDD is 8-digit format (though it should already be)
            merged_df['YYYYMMDD'] = merged_df['YYYYMMDD'].astype(str).str.zfill(8)
            
            # Save to new CSV file
            merged_df.to_csv(output_path, index=False, float_format='%.10f')  # Preserve precision
            
            print(f"Processed {filename} - saved to {output_path}")

# Example usage
input_folder = 'D://a//master//Earthquake-US//Data//20190706-500//ProcessedData//step3_renamed_files'
output_folder = 'D://a//master//Earthquake-US//Data//20190706-500//PosData-7'
process_csv_files(input_folder, output_folder)